# Section 3: Cortex AI Observability & Alerting Demo
**ACME IM | Distribution Insights**

Demonstrates DMFs, Snowflake Alerts, and Event Table telemetry.

> Note: `SNOWFLAKE.ACCOUNT_USAGE.ALERT_HISTORY` has up to 2-hour lag.
> If the alert table is empty, check again after alerts have run.

In [ ]:
import os, tomllib, json, urllib.request, urllib.error
from cryptography.hazmat.primitives.serialization import load_pem_private_key
from snowflake.snowpark import Session
from snowflake.cortex import Complete
import snowflake.connector
import pandas as pd

# Load keypair auth from connections.toml (works locally; get_active_session() used in SiS)
_CONN_NAME = os.environ.get('SNOWFLAKE_DEFAULT_CONNECTION_NAME', 'your_connection')
with open(os.path.expanduser('~/.snowflake/connections.toml'), 'rb') as f:
    _cfg = tomllib.load(f)[_CONN_NAME]

_key_path = os.path.expanduser(_cfg['private_key_path'])
with open(_key_path, 'rb') as f:
    _private_key = load_pem_private_key(f.read(), password=None)

_params = {k: v for k, v in _cfg.items() if k != 'private_key_path'}
_params['private_key'] = _private_key

# Snowpark Session
session = Session.builder.configs(_params).create()
session.sql('USE WAREHOUSE INGEST').collect()
session.sql('USE DATABASE ANALYTICS_DEV_DB').collect()
session.sql('USE SCHEMA ANALYTICS_DEV_DB.DISTRIBUTION').collect()

# REST token for Cortex Analyst
_SF_HOST = f"{_cfg['account']}.snowflakecomputing.com"
_raw_conn = snowflake.connector.connect(**_params)
_SF_TOKEN = _raw_conn.rest.token

print(f'Connected: {session.get_current_account()}')
print(f'Context: {session.get_current_database()}.{session.get_current_schema()}')


In [ ]:
# Demo 1: Data quality monitoring (DMF results, last 24 hours)
# Schema filter avoids showing unrelated tables from other projects
dmf_df = session.sql("""
    SELECT TABLE_NAME, METRIC_NAME, VALUE, MEASUREMENT_TIME,
           CASE WHEN (METRIC_NAME LIKE '%NULL%' OR METRIC_NAME LIKE '%DUPLICATE%')
                     AND VALUE > 0
                THEN 'WARNING' ELSE 'OK' END AS health_status
    FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
    WHERE TABLE_DATABASE = 'ANALYTICS_DEV_DB'
      AND TABLE_SCHEMA IN ('STAGING', 'DISTRIBUTION')
      AND MEASUREMENT_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
    ORDER BY MEASUREMENT_TIME DESC
""").to_pandas()
print(f'DMF Results (last 24h): {len(dmf_df)} measurements')
print(dmf_df.to_string(index=False) if len(dmf_df) else '  No DMF results yet — DMFs run on first scheduled cycle')


In [ ]:
# Demo 2: Alert history
# Correct column names: NAME, STATE, SCHEMA_NAME (not ALERT_NAME, CONDITION_QUERY_STATUS)
alert_df = session.sql("""
    SELECT NAME, SCHEMA_NAME, SCHEDULED_TIME, STATE
    FROM SNOWFLAKE.ACCOUNT_USAGE.ALERT_HISTORY
    WHERE SCHEMA_NAME = 'DISTRIBUTION'
      AND SCHEDULED_TIME >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
    ORDER BY SCHEDULED_TIME DESC LIMIT 20
""").to_pandas()
print('ALERT HISTORY (last 6 hours):')
if len(alert_df) == 0:
    print('  No rows — ACCOUNT_USAGE has up to 2-hour lag; check again later')
else:
    print(alert_df.to_string(index=False))
